# Portfolio Optimization: Markowitz Efficient Frontier and Capital Allocation Line
This notebook retrieves historical data, calculates optimal portfolio weights and identifies Tangency Portfolio by maximizing Sharpe ratio.

### Assets and time Horizon
* **Assets** AAPL, MSFT, IBM, KO, PG, WMT, JNJ, PFE, XOM, CAT, JPM, BAC
* **Timeframe** 1990-01-01 to 2026-04-18
* **Interval** Quarterly

# Importing libraries

In [2]:
import pandas as pd
import numpy as np
import yfinance as yf
from scipy.optimize import minimize
import matplotlib
import matplotlib.pyplot as plt
from datetime import datetime
datetime_format = "%Y-%m-%d"

# Getting yahoo data
Retrieving data via `yfinance`.

In [3]:
tickers_symbols = "AAPL MSFT IBM KO PG WMT JNJ PFE XOM CAT JPM BAC"
num_assets = len(tickers_symbols.split())
tickers = yf.Tickers(tickers_symbols)
start = "1990-01-01"
end = "2026-04-18"
years = (datetime.strptime(end, datetime_format) - datetime.strptime(start, datetime_format)).days / 365.25
data = yf.download(tickers=tickers_symbols, interval='3mo', start=start, end=end, group_by='tickers', progress=False, auto_adjust=True)
data = data.sort_index(axis=1, level=[0,1], ascending=[True,False])

# Data preprocessing
Calculating the quarterly percentage returns and changes. Isolating daily returns, aggregate returns, daily changes and aggregate returns into a `data` DataFrame.

In [4]:
tickers = data.columns.levels[0]

qt_returns = data.xs(key='Close', axis=1, level=1).pct_change(1)
qt_change = qt_returns + 1
qt_agg_change = qt_change.cumprod()
qt_agg_returns = qt_agg_change - 1

qt_returns.columns = pd.MultiIndex.from_product([qt_returns.columns, ['Quarterly_Returns']])
qt_change.columns = pd.MultiIndex.from_product([qt_change.columns, ['Quarterly_Change']])
qt_agg_change.columns = pd.MultiIndex.from_product([qt_agg_change.columns, ['Quarterly_Aggregate_Change']])
qt_agg_returns.columns = pd.MultiIndex.from_product([qt_agg_returns.columns, ['qt_agg_returns']])

data = data.sort_index(axis=1)
data = pd.concat([data, qt_returns, qt_change, qt_agg_change, qt_agg_returns], axis=1)
data.dropna(inplace=True)

# Covariance matrix of quarterly returns.

In [5]:
cov = data.xs('Quarterly_Returns', axis=1, level=1).cov()

# Total variance function
Total variance is given by formula
$$
\sigma_p^2 = \boldsymbol{w}^{\top}\Sigma\boldsymbol{w},
$$
where $\boldsymbol{w}$ is a vector of weights and $\Sigma$ is a covariance matrix.

In [6]:
def total_variance(weights):
    variance = np.transpose(weights).dot(cov.dot(weights))
    return variance

# Total expected return function
Total expected return is given by formula
$$
E(R_p) = \boldsymbol{w}^{\top}\boldsymbol{\mu},
$$
where $\boldsymbol{\mu}$ is a vector of returns of singular stocks. 

In [7]:
mean_returns = data.xs('Quarterly_Returns', axis=1, level=1).mean()
mean_returns.rename('Average_quarterly_returns', inplace=True)

variance = pd.DataFrame({'Ticker':tickers,'Variance' :np.diag(cov)})
variance.set_index('Ticker', inplace=True)

df_risk_expect_return = pd.concat([mean_returns, variance], axis=1)

def total_expected_return(weights):

        return np.transpose(weights).dot(df_risk_expect_return['Average_quarterly_returns'].values) 

# The Efficient Frontier Optimization
Calculating the minimum variance for 50 evenly spaced target returns.\
**Constraints:**
* Long-only weights;
* Fully invested portfolio.

In [8]:
Return_min = min(df_risk_expect_return['Average_quarterly_returns'].values)
Return_max = max(df_risk_expect_return['Average_quarterly_returns'].values)

target_returns = np.linspace(Return_min, Return_max, num=50)
initial_guess = np.repeat(1/num_assets, num_assets)
bounds = tuple((0.0, 1.0) for _ in range(num_assets)) 
efficient_frontier = []
for ret in target_returns:
    constraint = (
        {'type': 'eq','fun': lambda weights: sum(weights) -1}, 
        {'type': 'eq','fun': lambda weights: total_expected_return(weights) - ret}
    )
         
    res = minimize(fun=total_variance,
                   x0=initial_guess,
                   method='SLSQP',
                   bounds = bounds,
                   constraints=constraint)
    efficient_frontier.append(res.x)

# The Sharpe Ratio
Identifying the portfolio that yields the maximum risk-adjusted return by minimizing the negative Sharpe Ratio. **The Sharpe** ratio is given by
$$
S = \frac{R_j - R_f}{\sigma_j},
$$
where $S$ is a Sharpe ratio, $R_j$ is a rate of return of a portfolio, $R_f$ is a risk free rate of return and $\sigma_j$ is the standard deviation of the portfolio. I assumed the annual risk free rate to be 4%.

In [9]:
risk_free_rate = ( 0.04 + 1) ** (1/4) - 1


def neg_sharpe(params):

    risk_return = total_expected_return(params)
    risk_variance = total_variance(params)

    S = (-1) * (risk_return - risk_free_rate) / np.sqrt(risk_variance)

    return S

constraint_sharpe = {'type': 'eq', 'fun': lambda weights: sum(weights) - 1}
bounds_sharpe = tuple((0.0, 1.0) for _ in range(num_assets))
initial_guess = tuple(1/num_assets for _ in range(num_assets))

res_neg_sharpe_params = minimize(
    fun = neg_sharpe,
    x0=initial_guess,
    bounds = bounds,
    constraints=constraint_sharpe,
    method='SLSQP'
)

# Visualization
Plotting the Efficient Frontier, the Capital Allocation Line and single stocks.

In [ ]:
def line(a, x, b):
    return a * x + b


# plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(10, 6), dpi=100)

variances = [total_variance(params) for params in efficient_frontier]
returns = [total_expected_return(params) for params in efficient_frontier]
x_data = [np.sqrt(total_variance(params)) * 100 for params in efficient_frontier]
y_data = [(total_expected_return(params)) * 100 for params in efficient_frontier]
x_line_data = np.linspace(start=0, stop=20)

ax.plot(x_data, y_data, color="#1f77b4", linewidth=2.5, label="Efficient Frontier")
ax.scatter(
    np.sqrt(df_risk_expect_return["Variance"]) * 100,
    df_risk_expect_return["Average_quarterly_returns"] * 100,
    label="Singular stocks",
)
ax.plot(
    x_line_data,
    line(-neg_sharpe(res_neg_sharpe_params.x), x_line_data, risk_free_rate * 100),
    label="Capital Allocation Line",
)
print(f"Sharpe Ratio: {-neg_sharpe(res_neg_sharpe_params.x):.2f}")
print(f"Total expected return [%] with maximum Sharpe Ratio: {total_expected_return(res_neg_sharpe_params.x) * 100:.2f}"
)
print(
    "Standard deviation [%] for maximum Sharpe Ratio:",
    total_variance(res_neg_sharpe_params.x) ** (0.5) * 100,
)
ax.set_title(
    "Portfolio optimization: The Efficient Frontier",
    fontsize=16,
    fontweight="bold",
    pad=16,
)
ax.set_xlabel("Quarterly risk (standard deviation) [%]", fontsize=12, labelpad=10)
ax.set_ylabel("Expected quarterly rate of return [%]", fontsize=12, labelpad=10)

ax.tick_params(axis="both", which="major", labelsize=10)
ax.grid(True, linestyle="--", alpha=0.5)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(1.2)
ax.spines["bottom"].set_linewidth(1.2)

ax.legend(loc="upper left", frameon=True, fontsize=12)
fig.tight_layout()
plt.show()

SyntaxError: unterminated f-string literal (detected at line 27) (2099766872.py, line 27)